In [1]:
!git config --global user.email "beglossy@yonsei.ac.kr"
!git config --global user.name "beglossy-cmyk"

from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/beglossy-cmyk/gpt2.0_by_InaYoon.git
%cd gpt2.0_by_InaYoon
print("완료!")

Cloning into 'gpt2.0_by_InaYoon'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 17 (delta 6), reused 10 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 11.65 KiB | 2.33 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/gpt2.0_by_InaYoon
완료!


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# 셰익스피어 데이터
if not Path("input.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

print(f"vocab_size: {vocab_size}")
print(f"데이터 길이: {len(data)}")

vocab_size: 65
데이터 길이: 1115394


In [3]:
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 32
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

xb, yb = next(iter(loader))
print(f"xb.shape: {xb.shape}")  # (64, 32)
print(f"yb.shape: {yb.shape}")  # (64, 32)

# 실제로 어떻게 생겼는지 확인
print(f"\n입력 예시: {''.join([itos[i.item()] for i in xb[0]])}")
print(f"정답 예시: {''.join([itos[i.item()] for i in yb[0]])}")

xb.shape: torch.Size([64, 32])
yb.shape: torch.Size([64, 32])

입력 예시: t I have been;
Not my deserts, b
정답 예시:  I have been;
Not my deserts, bu


In [4]:
class MinimalSequenceModel(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        tok = self.token_embedding(x)           # 각 글자 → 64차원 벡터
        pos = self.position_embedding(torch.arange(T, device=x.device))  # 위치 정보
        h = tok + pos                           # 글자 + 위치 합치기
        logits = self.lm_head(h)               # 64 → 65 (vocab_size)
        return logits

model = MinimalSequenceModel(vocab_size, block_size)
logits = model(xb)
print(f"logits.shape: {logits.shape}")  # (64, 32, 65)

logits.shape: torch.Size([64, 32, 65])


In [5]:
def sequence_cross_entropy(logits, targets):
    # logits: (B, T, V) → (B, V, T) 로 변환해야 PyTorch loss 계산 가능
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 device: {device}")

model = MinimalSequenceModel(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(50):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    if epoch % 10 == 0 or epoch == 49:
        print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

사용 device: cpu
epoch  0 | train loss 3.0441
epoch 10 | train loss 2.4626
epoch 20 | train loss 2.4621
epoch 30 | train loss 2.4559
epoch 40 | train loss 2.4567
epoch 49 | train loss 2.4567


In [6]:
@torch.no_grad()
def sample(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]  # 마지막 위치 예측만 사용
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300))

ROMEO:
PENARAs r ha-se heen hmanghat y onsothapes alll hata t

Mouseensee harspld him.
Hend, as t to.
cothailofe,
Watodenour phe
RIs TRGLUCheachile nd it t cive,

BHewndume lenelaye ondss wondss otompre, andy o n

Bealiurd ngedothawin:
WARDuf ce thaness wharo,'d in w? t te, tr meld,
GHESt'd we f g m dess 


In [ ]:
from google.colab import userdata, _message
import json

nb = _message.blocking_request('get_ipynb', request='', timeout_sec=120)
with open('/content/gpt2.0_by_InaYoon/notebook_04_gpt_dataset.ipynb', 'w') as f:
    json.dump(nb['ipynb'], f)

token = userdata.get('GITHUB_TOKEN')
%cd /content/gpt2.0_by_InaYoon
!git remote set-url origin https://{token}@github.com/beglossy-cmyk/gpt2.0_by_InaYoon.git
!git add notebook_04_gpt_dataset.ipynb
!git commit -m "Add notebook_04: GPT-style Dataset + Minimal Sequence Model"
!git push origin main